# Qdrant quantization smoke workflow

This notebook checks the repository workflow on synthetic 384-dimensional vectors. It creates real collections on a Qdrant **v1.19.2** server, builds exact float32 ground truth, runs the smoke configurations, and summarizes the measurements.

**Required:** a source ZIP downloaded from the private repository while signed into GitHub, and an experimental Qdrant endpoint reachable over HTTPS from this Colab runtime with API-key authentication. Set up the endpoint before running the notebook. The VM Compose service binds to loopback and is not directly reachable from Colab. Do not expose an unauthenticated VM port.

No Cohere key, paid embedding call, model download, or BEIR download is needed. Synthetic results check the pipeline. Colab timings include its network path and are not the VM benchmark results.

Run this notebook in Google Colab. It installs locked repository dependencies using uv and needs internet access. Every failed command stops execution. No unavailable measurement is converted into success.

In [ ]:
import getpass
import hashlib
import io
import json
import os
import shutil
import stat
import subprocess
import tempfile
import zipfile
from datetime import UTC, datetime
from pathlib import Path
from urllib.parse import urlparse

from google.colab import files

## Upload source and install the locked project

While signed into GitHub, open `qdrant-labs/quantization-comparison-bench` at the desired commit and download its source ZIP. Upload only that source archive below; no access token is needed in the notebook. The source archive checksum is saved with the outputs. Record the GitHub commit alongside it because a ZIP has no `.git` history.

The installer is pinned to uv 0.12.3. `uv sync` supplies the project's required Python when necessary.

In [ ]:
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one repository source ZIP.")
archive_name, archive_bytes = next(iter(uploaded.items()))
source_sha256 = hashlib.sha256(archive_bytes).hexdigest()
source_directory = Path(tempfile.mkdtemp(prefix="qbench-source-"))
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for member in archive.infolist():
        target = (source_directory / member.filename).resolve()
        if not target.is_relative_to(source_directory):
            raise ValueError(
                "Archive contains a path outside its extraction directory."
            )
        if stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError("Source archive must not contain symbolic links.")
    archive.extractall(source_directory)
projects = list(source_directory.rglob("pyproject.toml"))
if len(projects) != 1:
    raise ValueError("Expected one source project with a pyproject.toml.")
project = projects[0].parent
source_commit = input("GitHub commit SHA for this source archive: ").strip()
if len(source_commit) != 40 or any(c not in "0123456789abcdef" for c in source_commit):
    raise ValueError("Enter the complete 40-character lowercase Git commit SHA.")
print(f"Source ZIP SHA256: {source_sha256}")

uv_bin = shutil.which("uv")
if uv_bin is None:
    subprocess.run(
        [
            "bash",
            "-o",
            "pipefail",
            "-c",
            "curl -LsSf https://astral.sh/uv/0.12.3/install.sh | sh",
        ],
        check=True,
    )
    uv_bin = str(Path.home() / ".local" / "bin" / "uv")
subprocess.run([uv_bin, "--version"], check=True)
subprocess.run([uv_bin, "sync", "--locked"], cwd=project, check=True)

## Connect to the experimental server

Enter the URL without credentials or query parameters. The API key prompt is hidden; it is kept only in the runtime environment and is not written to a cell or repository file. The run creates and removes its own experiment collections. Use a dedicated server so other collections and activity do not affect the measurements.

In [ ]:
endpoint = input("Experimental Qdrant HTTPS URL: ").strip().rstrip("/")
parsed = urlparse(endpoint)
if parsed.scheme != "https" or not parsed.netloc:
    raise ValueError("Provide an HTTPS endpoint reachable from this runtime.")
if parsed.username or parsed.password or parsed.query or parsed.fragment:
    raise ValueError("Keep credentials and query parameters out of the URL.")
api_key = getpass.getpass("Qdrant API key: ")
if not api_key:
    raise ValueError("This external-endpoint workflow requires an API key.")
os.environ["QDRANT_URL"] = endpoint
os.environ["QDRANT_API_KEY"] = api_key
del api_key


def run_qbench(*arguments: str) -> None:
    subprocess.run([uv_bin, "run", "qbench", *arguments], cwd=project, check=True)


run_qbench("doctor")

## Build the fixture and exact reference

The deterministic fixture has 256 documents, 32 queries, and synthetic relevance labels. Exact rankings are derived from the saved float32 vectors; neither these labels nor the corpus represent a real retrieval task.

In [ ]:
data_dir = "data/colab-synthetic"
run_qbench("fixture", "--output", data_dir)
run_qbench("verify", "--data", data_dir)
run_qbench("ground-truth", "--data", data_dir)

## Run and summarize

The smoke configuration limits work so the API workflow is easy to inspect. It is not the full method/configuration matrix. Read `configs/benchmark.yaml` and `docs/protocol.md` before running the real experiment on the VM.

In [ ]:
run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
result_dir = f"results/colab-smoke-{run_id}"
run_qbench(
    "run",
    "--config",
    "configs/smoke.yaml",
    "--data",
    data_dir,
    "--output",
    result_dir,
)
run_qbench("summarize", "--input", result_dir)
summary_path = project / result_dir / "summary.json"
print(json.dumps(json.loads(summary_path.read_text()), indent=2))

## Preserve outputs and clear the runtime key

Download the result archive from the notebook file browser before the runtime expires. It contains the manifest, raw query records, summary, collection/resource metadata, and the source archive checksum with the commit you supplied. A source ZIP cannot independently verify its Git commit; retain the original download and verify it against the repository before publishing.

Never publish endpoint credentials. A real paper comparison also needs held-out BEIR relevance, repeated index builds, host memory/storage measurements, and controlled cache conditions.

In [ ]:
(project / result_dir / "source-archive.json").write_text(
    json.dumps(
        {
            "filename": archive_name,
            "sha256": source_sha256,
            "user_supplied_git_commit": source_commit,
        },
        indent=2,
    )
)
archive_path = shutil.make_archive(
    str(Path.cwd() / f"qbench-smoke-{run_id}"),
    "zip",
    root_dir=project / result_dir,
)
os.environ.pop("QDRANT_API_KEY", None)
print(f"Results archive: {archive_path}")